In [1]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient, models
from pathlib import Path
import uuid
import gc

In [2]:
encoder = SentenceTransformer(
    "Qwen/Qwen3-Embedding-0.6B",
    device="cpu"
)

client = QdrantClient(path="../saves/VScatalogo") # Carrega vectorstore em disco

In [3]:
file_path = "/home/marcos_manhaes/Documentos/edital_petrobras.pdf"
#file_path = "/home/migueldcarvalho/Downloads/UERJ_2026.pdf"
collection_name = Path(file_path).name

In [4]:
def embed_texts(texts, encoder, batch_size=5):
    vectors = []
    print(f"Tamanho do texto: {len(texts)}")
    for i in range(0, len(texts), batch_size):
        print(i)
        batch = texts[i:i + batch_size]
        emb = encoder.encode(batch, convert_to_numpy=True)
        vectors.extend(emb)
        del batch, emb
        gc.collect()
    return vectors

def external_doc(state):
    client = state["client"]
    encoder = state["encoder"]


    loader = PyPDFLoader(file_path)
    docs = loader.load()

    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1500,
        chunk_overlap=300,
        length_function=len,
        separators=["\n\n", "\n", ".", " ", ""]
    )

    chunks = text_splitter.split_documents(docs)
    texts = [chunk.page_content for chunk in chunks]

    vectors = embed_texts(texts, encoder, batch_size=5)

    if not client.collection_exists(collection_name=collection_name):
        client.create_collection(
            collection_name=collection_name, # nome da coleção
            vectors_config=models.VectorParams(
            size=encoder.get_sentence_embedding_dimension(),  # Vector size is defined by used model
            distance=models.Distance.COSINE, # Metrica de similaridade
        ),
)

    points = []
    for i, chunk in enumerate(chunks):
        points.append(
            models.PointStruct(
                id=str(uuid.uuid4()),
                vector=vectors[i].tolist(),
                payload={
                    "page_content": chunk.page_content,
                    "metadata": chunk.metadata,
                    "chunk_index": i
                }
            )
        )

    client.upsert(
        collection_name=collection_name,
        points=points
    )

In [5]:
query = "Qual é o dia de volta as aulas no primeiro periodo?"

state = {
    "client": client,
    "encoder": encoder
}

if not client.collection_exists(collection_name=collection_name):
        external_doc(state)


In [ ]:
query = "Quero informações sobre os requisitos necessário para enfermagem do trabalho"
#query = "I would like information regarding the requirements for occupational nursing."
#query = "Quais são os requisitos de formação para a ênfase em Manutenção – Caldeiraria no concurso da Petrobras?"

reranker = CrossEncoder('BAAI/bge-reranker-base')

hits = client.search(
    collection_name=collection_name,
    query_vector=encoder.encode(query).tolist(),
    limit=10
)

pairs = [[query, hit.payload['page_content']] for hit in hits]
scores = reranker.predict(pairs)
scored_hits = sorted(zip(scores, hits), key=lambda x: x[0], reverse=True)

for score, hit in scored_hits:
    print(f"Score (cross-encoder): {score}")
    print(hit.payload['page_content'])
    print()

Score (cross-encoder): 0.9505084753036499
(Lei federal nº 7.498/1986) e suas atualizações. 10 O código de ética dos profissionais de 
enfermagem. Fundamentos de enfermagem.  
ÊNFASE 2: INSPEÇÃO DE EQUIPAMENTOS E INSTALAÇÕES – BLOCO I: 1 Eletroquímica. 2 Desenho 
técnico. 3 Dilatação térmica. 4 Sistema Internacional e conversão de unidades. 5 Estática. 6 
Dinâmica. 7 Metrologia. 8 Funções químicas. 9 Medição de temperatura e suas escalas. 10 
Conversão de unidades. BLOCO II: 1 Aço Carbono — diagrama de equilíbrio. 2 Hidrostática. 3 
Eletricidade básica. 4 Ondas mecânicas e eletromagnéticas. 5 Reações de óxidoredução. 6 Noções 
sobre ensaios não destrutivos. BLOCO III:  1 Transferência de calor. 2 Estequiometria. 3 
Hidrocarbonetos. 4 Soldagem — eletrodo revestido e TIG. 5 Processos de fabricação. 6 Mudanças 
de estado. 7 Calorimetria. 8 Noções sobre corrosão.  
ÊNFASE 3: LOGÍSTICA DE TRANSPORTES – CONTROLE – BLOCO I: 1 Logística, armazenagem e

Score (cross-encoder): 0.3059861361980438
